In [10]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import src.features as F

df = pd.read_parquet("../data/processed/clean.parquet")
feats = F.build_features(df)

X = feats[F.NUMERIC_FEATURES + F.CATEGORICAL_FEATURES]
print("Shape:", X.shape)
print("Infinite values:", int(np.isinf(X[F.NUMERIC_FEATURES]).sum().sum()))
display(X.isna().sum()[lambda s: s > 0])

# Spearman correlation of each numeric feature with the target
corr = (feats[F.NUMERIC_FEATURES]
        .corrwith(feats["duration"], method="spearman")
        .sort_values(key=abs, ascending=False))
display(corr)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Shape: (197272, 25)
Infinite values: 0


subtotal                                          179
min_item_price                                    813
max_item_price                                    800
total_onshift_dashers                           16268
total_busy_dashers                              16268
total_outstanding_orders                        16291
estimated_store_to_consumer_driving_duration      526
available_dashers                               16288
busy_ratio                                      16288
outstanding_per_dasher                          16309
avg_item_price                                    179
price_range                                       813
estimated_total                                   526
dtype: int64

outstanding_per_dasher                          0.365598
estimated_total                                 0.287352
estimated_store_to_consumer_driving_duration    0.265841
subtotal                                        0.236294
total_outstanding_orders                        0.180724
num_distinct_items                              0.161315
max_item_price                                  0.160261
total_items                                     0.155865
price_range                                     0.148342
busy_ratio                                      0.142092
available_dashers                              -0.132032
hour                                            0.124354
total_busy_dashers                              0.105676
estimated_order_place_duration                  0.103484
avg_item_price                                  0.087754
total_onshift_dashers                           0.081159
day_of_week                                     0.067014
is_no_dashers_onshift          

In [11]:
zero = feats["total_onshift_dashers"] == 0
missing = feats["is_dasher_data_missing"] == 1
normal = ~zero & ~missing

print("Rows with onshift == 0:", int(zero.sum()))
print("Mean duration (s) -> zero-onshift:", round(feats.loc[zero, "duration"].mean()),
      "| normal:", round(feats.loc[normal, "duration"].mean()),
      "| dasher-missing:", round(feats.loc[missing, "duration"].mean()))
display(feats.loc[zero, ["total_busy_dashers", "total_outstanding_orders"]].describe().T)

display(feats.groupby("hour")["duration"].agg(["count", "mean", "median"]).round(0))
display(feats.groupby("day_of_week")["duration"].agg(["count", "mean"]).round(0))

Rows with onshift == 0: 3604
Mean duration (s) -> zero-onshift: 3331 | normal: 2843 | dasher-missing: 2842


,count,mean,std,min,25%,50%,75%,max
total_busy_dashers,3601.0,0.044432,0.834148,0.0,0.0,0.0,0.0,29.0
total_outstanding_orders,3604.0,0.118757,0.704683,0.0,0.0,0.0,0.0,13.0


,count,mean,median
hour,,,
6,40,4058.0,3651.0
7,536,3162.0,2842.0
8,2104,2837.0,2636.0
9,3412,2651.0,2454.0
10,5096,2444.0,2262.0
11,13531,2763.0,2589.0
12,15549,2642.0,2476.0
13,11455,2455.0,2289.0
14,8808,2506.0,2334.0


,count,mean
day_of_week,,
0,24572,2828.0
1,23919,2656.0
2,25252,2820.0
3,27037,2650.0
4,33078,3048.0
5,32823,2893.0
6,30591,2975.0


### Observations: marketplace and time features

**1. Zero-supply rows are real and slow.** 3,604 rows have `total_onshift_dashers == 0`, with almost no busy dashers or outstanding orders. They average 3,331 s against 2,843 s for normal rows, about 8 minutes longer. Setting their ratios to NaN would discard that signal, and median imputation (needed for Ridge) would make them look like average-load periods. **Decision:** divide by `max(onshift, 1)` so the ratios stay defined, and add `is_no_dashers_onshift` so the model can tell this regime apart.

**2. Dasher-missing rows are not special.** Rows with missing dasher data average 2,842 s, identical to normal rows (2,843 s), and `is_dasher_data_missing` has a Spearman correlation of about 0 with the target. The missingness looks uninformative. **Decision:** keep the flag for now and revisit it after SHAP in Phase 5.

**3. Hour and day of week are non-monotonic.** Mean delivery time peaks at the dinner rush (17:00-19:00, about 3,280 s at 18:00) against about 2,450 s at 10:00 and 13:00, with a smaller bump at 11:00. Friday is the slowest day (3,048 s) and Tuesday and Thursday are the fastest (about 2,650 s), so `is_weekend` is a weak summary of the weekly pattern. Spearman correlation understates both features because it only detects monotonic trends. **Decision:** trees use `hour` and `day_of_week` as they are. Ridge in Phase 4 needs one-hot or cyclical encoding for them.

**4. Signal strength.** The strongest single feature is `outstanding_per_dasher` (Spearman 0.37; it was 0.39 before zero-supply rows were included, because those rows have a ratio near 0 but are slow, which `is_no_dashers_onshift` now captures), then `estimated_total` (0.29), which beats either of its two components. No single feature is above 0.4, so the target is noisy and we should expect a modest ceiling on accuracy.